<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_1_Foundations_Python_and_Math/Month_02_Data_Analysis_with_Pandas_and_NumPy/Week_4_Data_Manipulation/Day_26_Time_Series_with_Pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/user-attachments/assets/2a4adf95-3d28-41e2-90d0-d06b2e9c8fa3" alt="Learn Data Science with Emmanuel Odenyire" width="180"/>

# Day 26: Time Series with Pandas

## Phase 1 | Month 2 | Week 4 | Day 26

**Goal:** Handle dates, times, and sequential data — used in finance, IoT, weather, and every business that tracks events over time.

### What You Will Learn
1. `DatetimeIndex` and `pd.to_datetime()`
2. Resampling: daily → monthly aggregation
3. Rolling windows and exponential moving averages
4. Date-based filtering and slicing
5. Time zone handling
6. Real case: Nairobi electricity consumption analysis

### Resources
- 📖 [Pandas Time Series Guide](https://pandas.pydata.org/docs/user_guide/timeseries.html)
- 🎥 [Time Series with Pandas — sentdex](https://www.youtube.com/watch?v=e0IZG6-BUSI)
- 📖 [Python for Finance (Time Series chapter)](https://www.oreilly.com/library/view/python-for-finance/9781491945360/)

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)

# DatetimeIndex fundamentals
dates_str = ['2024-01-15', '2024-03-01', '15-Feb-2024', '20240601']
print('Parsing various date formats:')
for ds in dates_str:
    try:
        print(f'  {ds:20} -> {pd.to_datetime(ds)}')
    except Exception as e:
        print(f'  {ds:20} -> Error: {e}')

# Date ranges
print('\nBusiness days in Jan 2024 (first 5):')
print(pd.date_range('2024-01-01', '2024-01-31', freq='B').tolist()[:5])

print('\nQuarterly dates in 2024:')
print(pd.date_range('2024-01-01', periods=4, freq='QS').tolist())

print('\nEvery 6 hours for one day:')
print(pd.date_range('2024-01-01', periods=5, freq='6H').tolist())

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)

# Nairobi electricity consumption (hourly for 365 days)
hours = pd.date_range('2024-01-01', periods=365*24, freq='H')
consumption = (
    100                                                     # base
    + 40 * np.sin(2 * np.pi * np.arange(365*24) / 24)     # daily cycle
    + 20 * np.sin(2 * np.pi * np.arange(365*24) / (24*7)) # weekly cycle
    + np.random.randn(365*24) * 15                          # noise
)
consumption = np.clip(consumption, 20, 250)

ts = pd.Series(consumption, index=hours, name='kwh')
print('Hourly electricity consumption (first 5):')
print(ts.head())

# Resample — hourly to daily
daily = ts.resample('D').mean()
print(f'\nHourly ({len(ts):,}) -> Daily ({len(daily):,})')
print('Daily mean (Jan):', daily['2024-01'].round(1).head())

# Resample — daily to monthly
monthly = ts.resample('ME').agg(['mean','min','max','sum']).round(2)
print('\nMonthly summary:')
print(monthly)

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)

# KCB bank stock prices (daily, 252 trading days)
prices = pd.Series(
    np.round(40 + np.cumsum(np.random.normal(0.05, 0.8, 252)), 2),
    index=pd.date_range('2024-01-02', periods=252, freq='B'),
    name='KCB_KES'
)

# Rolling statistics
rolling_mean = prices.rolling(20).mean()
rolling_std  = prices.rolling(20).std()
upper_band   = rolling_mean + 2 * rolling_std
lower_band   = rolling_mean - 2 * rolling_std

# Exponential Moving Average (weights recent data more)
ema20 = prices.ewm(span=20, adjust=False).mean()
ema50 = prices.ewm(span=50, adjust=False).mean()

# Date-based slicing
print('Q1 prices (Jan-Mar 2024):')
print(prices['2024-01':'2024-03'].describe().round(2))

print('\nBollinger Bands (last 5 days):')
bb = pd.DataFrame({'Price':prices,'Upper':upper_band,'Mid':rolling_mean,'Lower':lower_band})
print(bb.dropna().tail().round(2))

print('\nEMA crossover signal (EMA20 > EMA50 = BUY):')
ema_df = pd.DataFrame({'EMA20':ema20,'EMA50':ema50}).dropna()
ema_df['Signal'] = np.where(ema_df['EMA20'] > ema_df['EMA50'],'BUY','SELL')
print(ema_df.tail(5).round(2))

## Exercise: JKIA Airport Traffic Time Series

Analyse passenger traffic time series data.

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(2024)

# Monthly passenger traffic at JKIA (5 years: 2019-2023)
months = pd.date_range('2019-01', periods=60, freq='MS')
base = 500_000
covid_factor = np.where((months.year == 2020) & (months.month >= 4), 0.15,
              np.where((months.year == 2020) & (months.month < 4), 1.0,
              np.where((months.year == 2021), 0.5,
              np.where((months.year == 2022), 0.8, 1.0))))
seasonality  = 1 + 0.2 * np.sin(2 * np.pi * (months.month - 1) / 12)
passengers = pd.Series(
    (base * seasonality * covid_factor * (1 + np.random.randn(60)*0.05)).astype(int),
    index=months, name='passengers'
)

# 1. Resample to yearly and find the COVID impact (2020 vs 2019)
# 2. Compute 12-month rolling average
# 3. Find month with highest and lowest traffic for each year
# 4. Compute YoY (year-over-year) growth rate
# 5. Identify months with traffic > 600,000 (peak season)
# YOUR CODE HERE

## Summary

- `pd.to_datetime()` parses almost any date format
- `DatetimeIndex` enables powerful date-based slicing (`ts['2024-01']`)
- `resample()` aggregates to any time frequency (hourly → daily → monthly)
- `rolling()` computes sliding window statistics
- `ewm()` gives more weight to recent observations

**Next: Day 27 — Advanced Pandas Techniques**